# ⚡ Snippy (Unsloth Colab Version): Gemma 3 270M Fine-Tuning & LiteRT Export

This notebook uses **Unsloth** (2x faster fine-tuning, 80% less VRAM) on a **free Tesla T4 Google Colab** GPU to fine-tune **Gemma 3 270M** as **Snippy** — an in-browser JavaScript Code Snippet & Tool Calling Agent — and export it to Google LiteRT format for browser WebGPU deployment.

## Step 1: Install Unsloth & LiteRT Conversion Tools
Install Unsloth and Google LiteRT export packages.

In [ ]:
# Install Unsloth and LiteRT tools
!pip install -q unsloth
!pip install -q litert-torch litert-lm

## Step 2: Load Gemma 3 270M with Unsloth FastLanguageModel
Load `unsloth/gemma-3-270m-it` and configure Unsloth fast training mode.

In [ ]:
# Import Unsloth FIRST before torch/transformers
from unsloth import FastLanguageModel
import torch

MODEL_ID = "unsloth/gemma-3-270m-it"
max_seq_length = 2048

# 1. Load Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    dtype=None,
)

# 2. Configure PEFT / LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# 3. Enable Unsloth Fast Training Mode
FastLanguageModel.for_training(model)
print("✅ Gemma 3 270M Unsloth Model Ready for Fast Training!")

## Step 3: Load Snippy Training Dataset
Fetch `snippy_dataset.json` containing 50+ generic tool calling instruction examples.

In [ ]:
import json
import os
from datasets import Dataset

DATASET_PATH = "snippy_dataset.json"
if not os.path.exists(DATASET_PATH):
    !wget -q https://raw.githubusercontent.com/silasly/gemma-litert-snippy-demo/main/snippy_dataset.json

with open(DATASET_PATH, "r") as f:
    sample_data = json.load(f)

dataset = Dataset.from_list(sample_data)
print(f"✅ Loaded {len(dataset)} training examples for Snippy in Colab!")

## Step 4: Fine-Tune Snippy with SFTTrainer
Train Gemma 3 270M on Colab GPU.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=1,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="results",
        padding_free=False,  # Disables Triton JIT kernel compilation pause
    ),
)
trainer.train()
print("✅ Snippy Fine-Tuning Complete!")

## Step 5: Save Merged Model
Use Unsloth's 1-line 16bit merging method (`save_pretrained_merged`).

In [ ]:
# Enable evaluation/inference mode before saving
FastLanguageModel.for_inference(model)

OUTPUT_MERGED_DIR = "fine_tuned_gemma_merged"
model.save_pretrained_merged(OUTPUT_MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"✅ Merged 16bit checkpoint saved to {OUTPUT_MERGED_DIR}!")

## Step 6: Convert Model to LiteRT Format (`litert-torch export_hf`)
Export model to `.litertlm` container with INT8 dynamic quantization.

In [ ]:
!litert-torch export_hf \
  ./fine_tuned_gemma_merged \
  ./litert_output \
  -b True \
  -q dynamic_int8

## Step 7: Extract WebGPU FlatBuffer & Download Converted Models
Extract the `TFL3` FlatBuffer (`model.tflite`) for WebGPU browser runtime and trigger direct file downloads in Colab.

In [ ]:
import os
from google.colab import files

SOURCE_MODEL = "./litert_output/model.litertlm"

if os.path.exists(SOURCE_MODEL):
    # Extract TFLite FlatBuffer for WebGPU browser engine
    with open(SOURCE_MODEL, "rb") as f:
        data = f.read()
    pos = data.find(b"TFL3")
    if pos != -1:
        tflite_data = data[pos - 4 :]
        with open("model.tflite", "wb") as f:
            f.write(tflite_data)
        print("✅ Extracted model.tflite (WebGPU FlatBuffer)!")
        files.download("model.tflite")
    
    print("✅ Downloading model.litertlm...")
    files.download(SOURCE_MODEL)
else:
    print(f"⚠️ Source model not found at {SOURCE_MODEL}")